# Model 2 : Transformer based
### Step 1 : Flattening JSON
Converting the JSON files to
dataframes




In [129]:
import json
import pandas as pd
import os

def load_all_files(folder_path):
    rows = []

    for file in os.listdir(folder_path):
        if file.endswith(".json"):
            with open(os.path.join(folder_path, file), encoding="utf-8") as f:
                data = json.load(f)

            for item in data["data"]:
                for para in item["paragraphs"]:
                    context = para["context"]

                    for qa in para["qas"]:
                        question = qa["question"]
                        answer = qa["answers"][0]["text"]

                        rows.append({
                            "context": context,
                            "question": question,
                            "answer": answer
                        })

    return pd.DataFrame(rows)

In [130]:
train_df = load_all_files("train/")
test_df  = load_all_files("test/")

print("Train size:", len(train_df))
print("Test size:", len(test_df))

Train size: 168
Test size: 72


### Step 2 : Prepare input

In [131]:
train_df["input"] = train_df["question"] + " [SEP] " + train_df["context"]
test_df["input"]  = test_df["question"]  + " [SEP] " + test_df["context"]

train_df["answer_in"]  = "bstart " + train_df["answer"]
train_df["answer_out"] = train_df["answer"] + " bend"

test_df["answer_in"]  = "bstart " + test_df["answer"]
test_df["answer_out"] = test_df["answer"] + " bend"



train_df.head()

,context,question,answer,input,answer_in,answer_out
0,الفلاسفة الطبيعيون الأوائل مثل طاليس وأنكسيمنا...,ما الفكرة المشتركة بين الفلاسفة الطبيعيين الأو...,كل مادة من دول بتتحرك وبتتحول طول الوقت,ما الفكرة المشتركة بين الفلاسفة الطبيعيين الأو...,bstart كل مادة من دول بتتحرك وبتتحول طول الوقت,كل مادة من دول بتتحرك وبتتحول طول الوقت bend
1,هيرقليطس كان يرى أن النار تمثل التغير المستمر....,لماذا اختار هيرقليطس النار كأصل للعالم؟,بسبب حركتها المجنونة والمستمرة,لماذا اختار هيرقليطس النار كأصل للعالم؟ [SEP] ...,bstart بسبب حركتها المجنونة والمستمرة,بسبب حركتها المجنونة والمستمرة bend
2,رغم اختلاف الفلاسفة، إلا أنهم اتفقوا على فكرة ...,ماذا كان رأي الفلاسفة حول أهمية الحركة؟,الحركة هي عصب الوجود بالكامل,ماذا كان رأي الفلاسفة حول أهمية الحركة؟ [SEP] ...,bstart الحركة هي عصب الوجود بالكامل,الحركة هي عصب الوجود بالكامل bend
3,مع ازدياد نفوذ الساموراي، بدأ احترام الناس لهم...,لماذا أصبح الناس يخافون من الساموراي؟,امتلاك السلاح وفرض النظام بالقوة,لماذا أصبح الناس يخافون من الساموراي؟ [SEP] مع...,bstart امتلاك السلاح وفرض النظام بالقوة,امتلاك السلاح وفرض النظام بالقوة bend
4,عالم الأعصاب دانيال وولبرت قدم تفسيرًا مختلفًا...,ماذا يقول عالم الأعصاب دانيال وولبرت عن وظيفة ...,وهي إنتاج سلسلة من الحركات المعقدة,ماذا يقول عالم الأعصاب دانيال وولبرت عن وظيفة ...,bstart وهي إنتاج سلسلة من الحركات المعقدة,وهي إنتاج سلسلة من الحركات المعقدة bend


### Step 3 : Tokenisation

In [132]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

tokenizer_enc = Tokenizer(oov_token="<OOV>")
tokenizer_enc.fit_on_texts(train_df["input"])

tokenizer_dec = Tokenizer(oov_token="<OOV>")
tokenizer_dec.fit_on_texts(
    list(train_df["answer_in"]) + list(train_df["answer_out"])
)


In [133]:
max_len = 30
max_answer_len = 10

X_train = pad_sequences(
    tokenizer_enc.texts_to_sequences(train_df["input"]),
    maxlen=max_len, padding="post"
)

X_test = pad_sequences(
    tokenizer_enc.texts_to_sequences(test_df["input"]),
    maxlen=max_len, padding="post"
)

decoder_input = pad_sequences(
    tokenizer_dec.texts_to_sequences(train_df["answer_in"]),
    maxlen=max_answer_len, padding="post"
)

decoder_target = pad_sequences(
    tokenizer_dec.texts_to_sequences(train_df["answer_out"]),
    maxlen=max_answer_len, padding="post"
)

decoder_target = np.expand_dims(decoder_target, -1)

print(X_train.shape)
print(decoder_input.shape)
print(decoder_target.shape)


(168, 30)
(168, 10)
(168, 10, 1)


In [134]:
decoder_input = pad_sequences(decoder_input, maxlen=20, padding="post", value=0)
decoder_target = pad_sequences(decoder_target, maxlen=20, padding="post", value=0)



print(X_train.shape)
print(decoder_input.shape)
print(decoder_target.shape)

(168, 30)
(168, 20)
(168, 20, 1)


### Step 6 : Train/Validation Split

In [135]:
from sklearn.model_selection import train_test_split

X_train, X_val, decoder_input_train, decoder_input_val, decoder_target_train, decoder_target_val = train_test_split(
    X_train,
    decoder_input,
    decoder_target,
    test_size=0.1
)

### Step 7 : Build the Model + Compile

In [136]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Dense, Add, LayerNormalization, Dropout
from tensorflow.keras.models import Model
import numpy as np

# -----------------------
# SMALL MODEL SETTINGS
# -----------------------
embed_dim = 32
num_heads = 1
ff_dim = 64

max_len = min(30, X_train.shape[1])
max_answer_len = min(8, decoder_input_train.shape[1])

vocab_size_enc = len(tokenizer_enc.word_index) + 1
vocab_size_dec = len(tokenizer_dec.word_index) + 1


# -----------------------
# POSITIONAL ENCODING
# -----------------------
class PositionalEncoding(tf.keras.layers.Layer):
    def __init__(self, max_len, embed_dim):
        super().__init__()
        pos = np.arange(max_len)[:, np.newaxis]
        i = np.arange(embed_dim)[np.newaxis, :]
        angle_rates = 1 / np.power(10000, (2 * (i // 2)) / np.float32(embed_dim))
        angle_rads = pos * angle_rates

        pe = np.zeros((max_len, embed_dim))
        pe[:, 0::2] = np.sin(angle_rads[:, 0::2])
        pe[:, 1::2] = np.cos(angle_rads[:, 1::2])

        self.pos_encoding = tf.constant(pe[np.newaxis, ...], dtype=tf.float32)

    def call(self, x):
        return x + self.pos_encoding[:, :tf.shape(x)[1], :]


# -----------------------
# ENCODER
# -----------------------
encoder_inputs = Input(shape=(max_len,))

x = Embedding(vocab_size_enc, embed_dim, mask_zero=True)(encoder_inputs)
x = PositionalEncoding(max_len, embed_dim)(x)

attn = tf.keras.layers.MultiHeadAttention(
    num_heads=num_heads,
    key_dim=embed_dim
)(x, x)

x = Add()([x, attn])
x = LayerNormalization()(x)
x = Dropout(0.1)(x)

ff = Dense(ff_dim, activation="relu")(x)
ff = Dense(embed_dim)(ff)

encoder_outputs = LayerNormalization()(Add()([x, ff]))


# -----------------------
# DECODER
# -----------------------
decoder_inputs = Input(shape=(max_answer_len,))

y = Embedding(vocab_size_dec, embed_dim, mask_zero=True)(decoder_inputs)
y = PositionalEncoding(max_answer_len, embed_dim)(y)

# causal mask (prevents looking ahead)
mask = tf.linalg.band_part(tf.ones((max_answer_len, max_answer_len)), -1, 0)
mask = mask[tf.newaxis, tf.newaxis, :, :]

self_attn = tf.keras.layers.MultiHeadAttention(
    num_heads=num_heads,
    key_dim=embed_dim
)(y, y, attention_mask=mask)

y = Add()([y, self_attn])
y = LayerNormalization()(y)
y = Dropout(0.1)(y)

# cross attention (important!)
cross_attn = tf.keras.layers.MultiHeadAttention(
    num_heads=num_heads,
    key_dim=embed_dim
)(y, encoder_outputs)

y = Add()([y, cross_attn])
y = LayerNormalization()(y)
y = Dropout(0.1)(y)

# feed forward
ff = Dense(ff_dim, activation="relu")(y)
ff = Dense(embed_dim)(ff)

y = LayerNormalization()(Add()([y, ff]))


# -----------------------
# OUTPUT
# -----------------------
outputs = Dense(vocab_size_dec, activation="softmax")(y)


# -----------------------
# MODEL
# -----------------------
model = Model([encoder_inputs, decoder_inputs], outputs)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'positional_encoding_26' (of type PositionalEncoding) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'positional_encoding_27' (of type PositionalEncoding) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Model: "functional_13"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_26      │ (None, 30)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_26        │ (None, 30, 32)    │     58,272 │ input_layer_26[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_encodin… │ (None, 30, 32)    │          0 │ embedding_26[0][… │
│ (PositionalEncodin… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 30, 32)    │      4,224 │ positional_encod… │
│ (MultiHeadAttentio… │                   │            │ positional_encod… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_27      │ (None, 8)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_65 (Add)        │ (None, 30, 32)    │          0 │ positional_encod… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_27        │ (None, 8, 32)     │     20,480 │ input_layer_27[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 30, 32)    │         64 │ add_65[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_encodin… │ (None, 8, 32)     │          0 │ embedding_27[0][… │
│ (PositionalEncodin… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_43          │ (None, 30, 32)    │          0 │ layer_normalizat… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 8, 32)     │      4,224 │ positional_encod… │
│ (MultiHeadAttentio… │                   │            │ positional_encod… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_65 (Dense)    │ (None, 30, 64)    │      2,112 │ dropout_43[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_67 (Add)        │ (None, 8, 32)     │          0 │ positional_encod… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_66 (Dense)    │ (None, 30, 32)    │      2,080 │ dense_65[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 8, 32)     │         64 │ add_67[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_66 (Add)        │ (None, 30, 32)    │          0 │ dropout_43[0][0], │
│                     │                   │            │ dense_66[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_45          │ (None, 8, 32)     │          0 │ layer_normalizat… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 121,248 (473.62 KB)

 Trainable params: 121,248 (473.62 KB)

 Non-trainable params: 0 (0.00 B)

Step 8 : Model Training

In [137]:
model.fit(
    [X_train[:, :max_len], decoder_input_train[:, :max_answer_len]],
    decoder_target_train[:, :max_answer_len],
    validation_data=([X_val[:, :max_len], decoder_input_val[:, :max_answer_len]],
                     decoder_target_val[:, :max_answer_len]),
    batch_size=16,
    epochs=40
)

Epoch 1/40
10/10 ━━━━━━━━━━━━━━━━━━━━ 9s 99ms/step - accuracy: 0.1474 - loss: 6.2059 - val_accuracy: 0.2868 - val_loss: 6.0397
Epoch 2/40
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.3121 - loss: 5.8738 - val_accuracy: 0.2868 - val_loss: 5.8766
Epoch 3/40
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.3121 - loss: 5.6318 - val_accuracy: 0.2868 - val_loss: 5.7563
Epoch 4/40
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.3121 - loss: 5.3938 - val_accuracy: 0.2868 - val_loss: 5.6517
Epoch 5/40
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.3121 - loss: 5.1729 - val_accuracy: 0.2868 - val_loss: 5.5493
Epoch 6/40
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.3121 - loss: 4.9596 - val_accuracy: 0.2868 - val_loss: 5.4524
Epoch 7/40
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 51ms/step - accuracy: 0.3121 - loss: 4.7629 - val_accuracy: 0.2868 - val_loss: 5.3642
Epoch 8/40
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.3121 - loss: 4.5797 - val_accuracy: 0.2868 - v

### Step 10 : Evaluation + Prediction

In [145]:
def decode_sequence(input_seq):
    target_seq = np.zeros((1, max_answer_len))
    target_seq[0, 0] = tokenizer_dec.word_index["bstart"]
    decoded_words = []

    for i in range(1, max_answer_len):
        preds = model.predict([input_seq[:, :max_len], target_seq], verbose=0)
        next_token_idx = np.argmax(preds[0, i-1, :])
        next_word = tokenizer_dec.index_word.get(next_token_idx, "")
        if next_word == "bend" or next_word == "":
            break
        decoded_words.append(next_word)
        target_seq[0, i] = next_token_idx

    return " ".join(decoded_words)


# Test predictions
correct = 0
for i in range(len(X_test)):
    input_seq = X_test[i:i+1]
    true_answer = test_df.iloc[i]["answer"].strip()
    predicted = decode_sequence(input_seq)

    if i < 5:
        print("Question:", test_df.iloc[i]["question"])
        print("True Answer:", true_answer)
        print("Predicted:", predicted)
        print("------")

    if predicted.strip() == true_answer:
        correct += 1

accuracy = correct / len(X_test) * 100
print(f"\nTest Accuracy (Exact Match): {correct}/{len(X_test)} = {accuracy:.2f}%")

Question: ما الذي كشف عن قدرة أورسون ويلز على التأثير في الجمهور قبل دخوله السينما؟
True Answer: تقديم بث إذاعي تسبب في ذعر جماهيري واسع
Predicted: بسبب برمجة بيولوجية مرتبطة بالغدة البصرية
------
Question: كيف كانت طبيعة الإنتاج السينمائي في هوليوود قبل ظهور Citizen Kane؟
True Answer: كانت تعتمد على أنماط تقليدية ونهايات متوقعة
Predicted: لأنه لم يكن يمتلك أسطولًا بحريًا يدعم
------
Question: ما التغيير الأساسي الذي قدمه Citizen Kane في طريقة سرد القصة؟
True Answer: استخدام سرد غير خطي يبدأ من النهاية
Predicted: بسبب برمجة بيولوجية مرتبطة بالغدة البصرية
------
Question: كيف ساهم تعدد وجهات النظر في تشكيل صورة شخصية كين؟
True Answer: عرض الشخصية من زوايا مختلفة ومتضاربة
Predicted: في أذرعه
------
Question: ما الهدف من استخدام تقنية Deep Focus في الفيلم؟
True Answer: تمكين المشاهد من رؤية جميع عناصر المشهد بوضوح واختيار ما يركز عليه
Predicted: في أذرعه
------

Test Accuracy (Exact Match): 2/72 = 2.78%
